# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library, referencing all data elements with their `@id` identifiers for consistency and reproducibility.

### Dataset Source
The dataset is defined by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and includes clinicopathological and molecular variables for 77 cancer survivors with second primary colorectal cancer.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and record access using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
List available record sets (`cr:RecordSet`) in the dataset and the fields/columns for each, referenced by `@id`.

Below, we inspect the dataset to identify available record sets and their constituent fields.

In [ ]:
# Discover and list all available record sets and their field @ids

def list_record_sets_info(dataset):
    print("Available Record Sets:")
    all_record_sets = list(dataset.record_sets)
    if not all_record_sets:
        print("No record sets found in dataset metadata.")
        return []

    for rec_set in all_record_sets:
        print(f"\nRecord Set @id: {rec_set['@id']}")
        fields = rec_set.get('field', [])
        # Some fields may be single dict or list
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields @id:")
        if not fields:
            print("    None defined")
        for field in fields:
            print(f"    {field['@id']} ({field.get('name', '')})")
    return all_record_sets

# List all record sets (with their @id) and their fields
record_sets_info = list_record_sets_info(dataset)

# Collect record set @ids for later use
record_set_ids = [rec_set['@id'] for rec_set in record_sets_info]

## 3. Data Extraction
Load each record set into a DataFrame using their `@id`. For demonstration, we work with the primary record set containing the main data table. Please adjust the `record_set_id` as appropriate based on the above listing.

In [ ]:
# Extract records from each record set, referenced by their @id
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} records, columns:", df.columns.tolist())
        else:
            print("  No records found for this record set.")
    except Exception as e:
        print(f"  Could not load records: {e}")

# For demonstration, select the first populated record set for EDA
if dataframes:
    main_record_set_id = next(iter(dataframes.keys()))
    print(f"\nUsing {main_record_set_id} as the main record set for further exploration.")
    display_cols = dataframes[main_record_set_id].columns.tolist()
    print("\nAvailable columns (@id):", display_cols)
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular dataframes were loaded. Check schema configuration or dataset contents.")

## 4. Exploratory Data Analysis (EDA)
Perform exploratory operations such as filtering, normalization, and grouping. All columns are referenced by their field `@id`. Adjust `numeric_field_id` and `group_field_id` below depending on which fields are available in the main record set.

In [ ]:
# Select a numeric field and a grouping field by their @id from the previous cell's output
# Replace '@id_of_numeric_field' and '@id_of_group_field' with actual values present in your data

main_df = dataframes.get(main_record_set_id)
if main_df is not None:
    # Display some example columns for user selection
    print("Columns in the main record set:", list(main_df.columns))
    
    # Example of @id-based selection. Replace below with real @id fields:
    numeric_field_id = None
    group_field_id = None
    
    # Try to auto-select a numeric field
    for c in main_df.columns:
        if main_df[c].dtype.kind in 'biufc':  # int or float types
            numeric_field_id = c
            break
    # Try to auto-select a group field
    for c in main_df.columns:
        if c != numeric_field_id and main_df[c].nunique() < (len(main_df) // 2):
            group_field_id = c
            break
    
    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Group by another field if available
        if group_field_id:
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for analysis.")
else:
    print("Main DataFrame not available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with the group field, using their `@id` references.
Please update the `numeric_field_id` and `group_field_id` if selecting different fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, examine, process, and visualize data from a FAIR-compliant Croissant schema, referencing all data elements by their `@id` for reproducible science. Please adapt field and record set IDs as needed for your analysis tasks.